In [2]:
import json

import requests
from utils import *

This notebook explores streaming examples of OpenAI (apply_patch, shell) and Anthropic (text_editor, bash) native tools

## OpenAI

### Basic custom tool

In [61]:
tools = [
    {
        "type": "function",
        "name": "add",
        "description": "Add two numbers",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "number"},
                "b": {"type": "number"},
            },
            "required": ["a", "b"],
            "additionalProperties": False,
        },
        "strict": True,
    }
]

In [62]:
resp = requests.post(
    "https://api.openai.com/v1/responses",
    headers=openai_headers,
    json={
        "model": "gpt-5.4-nano",
        "store": False,
        "input": "Use the add tool to compute 3 + 2",
        "tools": tools,
        "stream": True,
    },
    stream=True,
)

resp

<Response [200]>

In [63]:
lines = list(resp.iter_lines(decode_unicode=True))

In [64]:
with open("openai_custom_tool_streaming_lines.txt", "w") as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 50), indent=2) + "\n"
        print(line, file=fp)

### Image Generation

In [71]:
resp = requests.post(
    "https://api.openai.com/v1/responses",
    headers=openai_headers,
    json={
        "model": "gpt-5.4-nano",
        "store": False,
        "input": "Generate an image of a happy golden retriever in a countryside field",
        "tools": [{"type": "image_generation"}],
        "stream": True,
    },
    stream=True,
)

resp

<Response [200]>

In [72]:
lines = list(resp.iter_lines(decode_unicode=True))

In [73]:
with open("openai_image_generation_streaming_lines.txt", "w") as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 50), indent=2) + "\n"
        print(line, file=fp)

### Shell Tool

In [6]:
resp = requests.post(
    "https://api.openai.com/v1/responses",
    headers=openai_headers,
    json={
        "model": "gpt-5.4-nano",
        "store": False,
        "input": "Use the shell tool to list all the files in the current directory",
        "tools": [{"type": "shell", "environment": {"type": "local"}}],
        "stream": True,
    },
    stream=True,
)

resp

<Response [200]>

In [7]:
lines = list(resp.iter_lines(decode_unicode=True))

In [11]:
with open("openai_shell_local_streaming_lines.txt", "w") as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 50), indent=2) + "\n"
        print(line, file=fp)

### ApplyPatch

#### Create File

In [17]:
resp = requests.post(
    "https://api.openai.com/v1/responses",
    headers=openai_headers,
    json={
        "model": "gpt-5.4-nano",
        "store": False,
        "input": "Write a simple print 'hello world' in main.py using the apply_patch tool",
        "tools": [{"type": "apply_patch"}],
        "stream": True,
    },
    stream=True,
)

resp

<Response [200]>

In [18]:
lines = list(resp.iter_lines(decode_unicode=True))

In [19]:
with open("openai_apply_patch_create_file_streaming_lines.txt", "w") as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 200), indent=2) + "\n"
        print(line, file=fp)

#### Update File operation

In [12]:
prompt = """
The user has the following files:
<BEGIN_FILES>
===== lib/fib.py
def fib(n):
    if n <= 1:
        return n
    return fib(n-1) + fib(n-2)

===== run.py
from lib.fib import fib

def main():
  print(fib(42))
<END_FILES>

You are a helpful coding assistant that should assist the user with whatever they
ask.

User query:
Help me rename the fib() function to fibonacci()
"""

In [13]:
resp = requests.post(
    "https://api.openai.com/v1/responses",
    headers=openai_headers,
    json={"model": "gpt-5.4-nano", "store": False, "input": prompt, "tools": [{"type": "apply_patch"}], "stream": True},
    stream=True,
)

resp

<Response [200]>

In [14]:
lines = list(resp.iter_lines(decode_unicode=True))

In [16]:
with open("openai_apply_patch_update_file_streaming_lines.txt", "w") as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 200), indent=2) + "\n"
        print(line, file=fp)

#### Delete File

In [20]:
resp = requests.post(
    "https://api.openai.com/v1/responses",
    headers=openai_headers,
    json={
        "model": "gpt-5.4-nano",
        "store": False,
        "input": "Delete the file main.py using the apply_patch tool",
        "tools": [{"type": "apply_patch"}],
        "stream": True,
    },
    stream=True,
)

resp

<Response [200]>

In [21]:
lines = list(resp.iter_lines(decode_unicode=True))

In [22]:
with open("openai_apply_patch_delete_file_streaming_lines.txt", "w") as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 200), indent=2) + "\n"
        print(line, file=fp)

# Anthropic

### Basic custom tool

In [67]:
tools = [
    {
        "name": "add",
        "description": "Add two numbers",
        "input_schema": {
            "type": "object",
            "properties": {
                "a": {"type": "number"},
                "b": {"type": "number"},
            },
            "required": ["a", "b"],
        },
    }
]

In [68]:
resp = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers=ant_headers,
    json={
        "model": "claude-sonnet-5",
        "max_tokens": 4096,
        "messages": [
            {
                "role": "user",
                "content": "Use the add tool to compute 3 + 2",
            }
        ],
        "tools": tools,
        "stream": True,
    },
    stream=True,
)
resp

<Response [200]>

In [69]:
lines = list(resp.iter_lines(decode_unicode=True))

In [70]:
with open("anthropic_custom_tool_streaming_lines.txt", "w") as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 50), indent=2) + "\n"
        print(line, file=fp)

### Text Editor

#### View command

In [32]:
resp = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers=ant_headers,
    json={
        "model": "claude-sonnet-5",
        "max_tokens": 4096,
        "messages": [
            {
                "role": "user",
                "content": "there's syntax error in my primes.py file. Can you help me fix it? Start by reading the file with your text_editor tool",
            }
        ],
        "tools": [{"type": "text_editor_20250728", "name": "str_replace_based_edit_tool", "max_characters": 10000}],
        "stream": True,
    },
    stream=True,
)
resp

<Response [200]>

In [33]:
lines = list(resp.iter_lines(decode_unicode=True))

In [37]:
with open("anthropic_text_editor_command_view_streaming_lines.txt", "w") as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 50), indent=2) + "\n"
        print(line, file=fp)

#### Create command

In [43]:
resp = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers=ant_headers,
    json={
        "model": "claude-sonnet-5",
        "max_tokens": 4096,
        "messages": [
            {
                "role": "user",
                "content": "Write a simple print 'hello world' in main.py using the text_editor tool. The file doesn't exist.",
            }
        ],
        "tools": [{"type": "text_editor_20250728", "name": "str_replace_based_edit_tool", "max_characters": 10000}],
        "stream": True,
    },
    stream=True,
)
resp

<Response [200]>

In [44]:
lines = list(resp.iter_lines(decode_unicode=True))

In [45]:
with open("anthropic_text_editor_command_create_streaming_lines.txt", "w") as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 50), indent=2) + "\n"
        print(line, file=fp)

#### str_replace command

In [39]:
prompt = """
The user has the following files:
<BEGIN_FILES>
===== lib/fib.py
def fib(n):
    if n <= 1:
        return n
    return fib(n-1) + fib(n-2)

===== run.py
from lib.fib import fib

def main():
  print(fib(42))
<END_FILES>

You are a helpful coding assistant that should assist the user with whatever they
ask.

User query:
Help me rename the fib() function to fibonacci()
"""

In [40]:
resp = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers=ant_headers,
    json={
        "model": "claude-sonnet-5",
        "max_tokens": 4096,
        "messages": [{"role": "user", "content": prompt}],
        "tools": [{"type": "text_editor_20250728", "name": "str_replace_based_edit_tool", "max_characters": 10000}],
        "stream": True,
    },
    stream=True,
)
resp

<Response [200]>

In [41]:
lines = list(resp.iter_lines(decode_unicode=True))

In [42]:
with open("anthropic_text_editor_command_str_replace_streaming_lines.txt", "w") as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 50), indent=2) + "\n"
        print(line, file=fp)

#### str_replace command

In [54]:
prompt = """
The user has the following files:
<BEGIN_FILES>
===== lib/fib.py
def fib(n):
    if n <= 1:
        return n
    return fib(n-1) + fib(n-2)

<END_FILES>

You are a helpful coding assistant that should assist the user with whatever they
ask.

User query:
Insert an example usage of the fib function at the end of lib/fib.py
You MUST use the text_editor tool with the `insert` command. You can't view the file. You MUST trust what I sent.
"""

In [55]:
resp = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers=ant_headers,
    json={
        "model": "claude-sonnet-5",
        "max_tokens": 4096,
        "messages": [{"role": "user", "content": prompt}],
        "tools": [{"type": "text_editor_20250728", "name": "str_replace_based_edit_tool", "max_characters": 10000}],
        "stream": True,
    },
    stream=True,
)
resp

<Response [200]>

In [56]:
lines = list(resp.iter_lines(decode_unicode=True))

In [57]:
with open("anthropic_text_editor_command_insert_streaming_lines.txt", "w") as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 50), indent=2) + "\n"
        print(line, file=fp)

## Bash tool

In [58]:
resp = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers=ant_headers,
    json={
        "model": "claude-sonnet-5",
        "max_tokens": 4096,
        "messages": [{"role": "user", "content": "List all Python files in the current directory with your bash tool"}],
        "tools": [{"type": "bash_20250124", "name": "bash"}],
        "stream": True,
    },
    stream=True,
)
resp

<Response [200]>

In [59]:
lines = list(resp.iter_lines(decode_unicode=True))

In [60]:
with open("anthropic_bash_tool_basic_streaming_lines.txt", "w") as fp:
    for line in lines:
        if not line.strip():
            continue
        if line.startswith("data:"):
            line = json.dumps(preview(json.loads(line[6:]), 50), indent=2) + "\n"
        print(line, file=fp)